In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [2]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

decoder_start_token_id = model.config.decoder_start_token_id
if decoder_start_token_id is None:
    decoder_start_token_id = tokenizer.pad_token_id

eos_token_id = tokenizer.eos_token_id
assert eos_token_id is not None, "Tokenizer must define eos_token_id for sequence scoring."

candidate_token_ids = {
    "yes": tokenizer("yes", add_special_tokens=False).input_ids + [eos_token_id],
    "no": tokenizer("no", add_special_tokens=False).input_ids + [eos_token_id],
}

print(model_name)
print("decoder_start_token_id:", decoder_start_token_id)
print("eos_token_id:", eos_token_id)
print({k: {"ids": v, "decoded": tokenizer.decode(v)} for k, v in candidate_token_ids.items()})


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


google/flan-t5-small
decoder_start_token_id: 0
eos_token_id: 1
{'yes': {'ids': [4273, 1], 'decoded': 'yes</s>'}, 'no': {'ids': [150, 1], 'decoded': 'no</s>'}}


In [3]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

def build_prompt(s1, s2):
    return (
        "Are these two sentences paraphrases? Answer yes or no.\n"
        f"Sentence 1: {s1}\n"
        f"Sentence 2: {s2}\n"
        "Answer:"
    )

prompts = [build_prompt(a, b) for a, b in zip(sent1, sent2)]

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())
print("sample_prompt:\n", prompts[0])


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
sample_prompt:
 Are these two sentences paraphrases? Answer yes or no.
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
Answer:


In [4]:
def score_candidate(enc, candidate_ids):
    batch_size = enc["input_ids"].shape[0]
    target_ids = torch.tensor(candidate_ids, dtype=torch.long, device=device).unsqueeze(0).expand(batch_size, -1)
    target_len = target_ids.shape[1]

    decoder_input_ids = torch.full(
        (batch_size, target_len),
        fill_value=decoder_start_token_id,
        dtype=torch.long,
        device=device,
    )
    if target_len > 1:
        decoder_input_ids[:, 1:] = target_ids[:, :-1]

    outputs = model(**enc, decoder_input_ids=decoder_input_ids)
    log_probs = torch.log_softmax(outputs.logits, dim=-1)
    token_log_probs = log_probs.gather(dim=-1, index=target_ids.unsqueeze(-1)).squeeze(-1)
    seq_log_probs = token_log_probs.sum(dim=-1)
    return seq_log_probs

batch_size = 32
preds = []
yes_scores_all = []
no_scores_all = []

with torch.inference_mode():
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i + batch_size]

        enc = tokenizer(
            batch_prompts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        yes_scores = score_candidate(enc, candidate_token_ids["yes"])
        no_scores = score_candidate(enc, candidate_token_ids["no"])

        batch_preds = (yes_scores > no_scores).long().cpu().numpy()

        preds.extend(batch_preds.tolist())
        yes_scores_all.extend(yes_scores.cpu().numpy().tolist())
        no_scores_all.extend(no_scores.cpu().numpy().tolist())

y_pred = np.array(preds)
yes_scores_all = np.array(yes_scores_all)
no_scores_all = np.array(no_scores_all)
print("done")


  0%|          | 0/13 [00:00<?, ?it/s]

done


In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.6495098039215687, 'f1': 0.6911447084233261}
                precision    recall  f1-score   support

not_paraphrase       0.47      0.81      0.59       129
    paraphrase       0.87      0.57      0.69       279

      accuracy                           0.65       408
     macro avg       0.67      0.69      0.64       408
  weighted avg       0.74      0.65      0.66       408



In [6]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "answer:", "yes" if int(y_pred[i]) == 1 else "no")
    print("yes_logprob:", float(yes_scores_all[i]), "no_logprob:", float(no_scores_all[i]))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0 answer: no
yes_logprob: -0.8191786408424377 no_logprob: -0.602909505367279
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 answer: no
yes_logprob: -1.6394611597061157 no_logprob: -0.2748245298862457
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 0 answer: no
yes_logprob: -1.0107052326202393 no_logprob: -0.45562002062797546


In [7]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("yes_logprob:", float(yes_scores_all[i]), "no_logprob:", float(no_scores_all[i]))


num_errors: 143
idx: 0
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0
yes_logprob: -0.8191786408424377 no_logprob: -0.602909505367279
idx: 3
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will decide in October whether to endorse a candidate before the primaries .
true: 1 pred: 0
yes_logprob: -0.821238100528717 no_logprob: -0.6100975871086121
idx: 7
sentence1: This integrates with Rational PurifyPlus and allows developers to work in supported versions of Java , Visual C # and Visual Basic .NET.
sentence2: IBM said the Rational products were also integrated with Rational PurifyPlus , which allows developers to work in Java , Visual C # and VisualBasic .Net.
true: 1 pred: 0
yes_logprob: -0.9792486429214478 no_logprob: -0.564846396446228


In [8]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": str(device),
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
}
summary


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'google/flan-t5-small',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.6495098039215687,
 'f1': 0.6911447084233261}